This notebooks concatenates all processed Spatial datasets into a single spatial AnnData ready for smoothing and downstream compilation.

Operation(s):

1. Concatenate Spatial Datasets
2. Combines individual `.var['IsCaptured']` into a dataframe in `.varm`

Output(s):

1. DATADIR/7_spatial/SP_concat.h5ad

Requirement(s):

1. DATADIR/7_spatial/SP*-**.h5ad

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings

from pathlib import Path

import pandas as pd
import anndata as ad
import scanpy as sc
import numpy as np

from dotenv import dotenv_values
from matplotlib import pyplot as plt

from hm3d_utils.utils import write_clean_h5ad
from project_utils import FileTracker

warnings.simplefilter(action='ignore', category=FutureWarning)
sc.set_figure_params(frameon=False, dpi=120, figsize=(5, 5))

In [3]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "7_spatial"

tracker = FileTracker(log_file=DATADIR/'tracked_files.toml')

In [4]:
filepaths = sorted(OUTDIR.glob('SP*-*.h5ad'))

adata_dict = {
    p.stem: sc.read_h5ad(p)
    for p in filepaths
}

In [5]:
captured_dict = {}
for k, v in adata_dict.items():
    pathmat = v.uns['pathmat']
    
    captured_dict[k] = v.var['IsCaptured']
    
    var = v.var[['gene_stable_ids', 'IsHVG']]

In [6]:
concat = ad.concat(
    adata_dict,
    axis=0
)

concat.var[['gene_stable_ids', 'IsHVG']] = var.loc[concat.var_names]
concat.varm['IsCaptured'] = pd.DataFrame(captured_dict).loc[concat.var_names]
concat.uns['pathmat'] = pathmat

In [7]:
outfile = OUTDIR / 'SP_concat.h5ad'
with tracker.register(outfile) as tf:
    tf.set_description('Concatenated spatial object (single cells/beads).')
    write_clean_h5ad(concat, tf.filepath)

In [8]:
concat

AnnData object with n_obs × n_vars = 1401313 × 56939
    obs: 'Dataset', 'Sample', 'Author', 'Technology', 'Donor', 'Sex', 'RegionID', 'RegionAcronym', 'RegionName', 'total_counts', 'total_hvg_counts', 'total_genes', 'IsHY'
    var: 'gene_stable_ids', 'IsHVG'
    uns: 'pathmat'
    obsm: 'P_path', 'X_embed', 'X_umap', 'X_umap_sc', 'coords_ccf_mm', 'coords_ccf_mm_wrap', 'coords_source', 'coords_voxels_25um', 'coords_voxels_25um_bounded'
    varm: 'IsCaptured'